# CP3 — K-Means County Risk Tier Segmentation
**rx-risk-pricer | Team 6 — Actuarial Risk Squad | ITCS 6100 Spring 2026**

### Why K-Means after Random Forest?
Random Forest gives 3,031 individual predicted rates.  
K-Means collapses those into 3 actionable pricing tiers.  
They operate in **sequence**, not in competition.

### Result
| Tier | Counties | Avg Predicted Rate | Premium Multiplier |
|------|----------|-------------------|-------------------|
| Low Risk | 974 | 2.57 | 0.85× |
| Medium Risk | 1,435 | 4.14 | 1.00× |
| High Risk | 622 | 6.34 | 1.25× |

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_ingestion import load_processed_cms, prepare_processed_cms
from dataset_assembly import build_model_datasets
from feature_engineering import engineer_features
from model_training import load_model, predict_counties
from clustering import find_optimal_k, assign_risk_tiers, summarize_tiers, get_top_high_risk

sns.set_theme(style='whitegrid')

## 1. Load Model & Generate Predictions

In [ ]:
processed_2020 = prepare_processed_cms('../data/raw/cms_2020.csv', '../data/processed/cms_2020_cleaned.csv', year=2020)
processed_2021 = prepare_processed_cms('../data/raw/cms_2021.csv', '../data/processed/cms_2021_cleaned.csv', year=2021)

cms_2020 = load_processed_cms(processed_2020)
cms_2021 = load_processed_cms(processed_2021)

df_train, df_test, cutoff = build_model_datasets(cms_2020, cms_2021)
df_raw = pd.concat([df_train, df_test], ignore_index=True)
df = engineer_features(df_raw)

model = load_model('../outputs/models/rf_opioid_pricer.pkl')
county_preds = predict_counties(model, df)
print(f'Generated predictions for {len(county_preds)} counties')
county_preds.head()

## 2. Elbow Curve — Validating k=3

In [ ]:
# This plot justifies our choice of k=3 clusters
find_optimal_k(county_preds)

## 3. Assign Risk Tiers

In [ ]:
tiered_df = assign_risk_tiers(county_preds)
summary = summarize_tiers(tiered_df)

## 4. County Distribution by Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
tier_counts = tiered_df['risk_tier'].value_counts().reindex(['LOW', 'MEDIUM', 'HIGH'])
colors = ['#4caf50', '#ff9800', '#f44336']
axes[0].pie(tier_counts, labels=tier_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('County Distribution by Risk Tier')

# Rate distribution by tier
for tier, color in zip(['LOW', 'MEDIUM', 'HIGH'], colors):
    subset = tiered_df[tiered_df['risk_tier'] == tier]['predicted_rate']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=tier)
axes[1].set_xlabel('Predicted Opioid Rate')
axes[1].set_ylabel('County Count')
axes[1].set_title('Rate Distribution by Tier')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/tier_distribution.png', dpi=150)
plt.show()

## 5. Top High-Risk Counties

In [ ]:
top_risk = get_top_high_risk(tiered_df, n=10)
print('\n=== Top 10 Highest Risk Counties ===')
print(top_risk.to_string(index=False))

## 6. Save Final Output

In [ ]:
output_path = '../outputs/predictions/county_risk_tiers.csv'
tiered_df.to_csv(output_path, index=False)
print(f'✓ County risk tiers saved to {output_path}')
print(f'\nFinal output shape: {tiered_df.shape}')
preview_cols = [
    col for col in ['county_name', 'predicted_rate', 'actual_rate', 'risk_tier', 'premium_multiplier']
    if col in tiered_df.columns
]
print(tiered_df[preview_cols].head(10))

## 7. Summary

- **974 Low Risk counties** → 0.85× premium (15% discount to grow in safe markets)
- **1,435 Medium Risk counties** → 1.00× baseline (47% of all US counties)
- **622 High Risk counties** → 1.25× surcharge (hedge against crisis-zone exposure)

The pure actuarial multiplier for High Risk would be 1.53× (6.34 ÷ 4.14).  
We chose **1.25×** as a conservative commercial decision — aggressive enough to hedge risk,  
but not so high that we price ourselves out of those markets entirely.